<a href="https://colab.research.google.com/github/uncannyeng01/yantaramalisonarprojesi_yolo11/blob/main/01_baseline_filtreler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/sonar_proje'
os.makedirs(BASE, exist_ok=True)

!pip install -q ultralytics bm3d opencv-python-headless

In [ ]:
!wget -q --show-progress -O /content/AquaScan-1K.zip "https://zenodo.org/records/18771165/files/AquaScan-1K.zip?download=1"
!unzip -q /content/AquaScan-1K.zip -d /content/aquascan
!find /content/aquascan -maxdepth 3 | head -20

In [ ]:
!apt-get -qq install aria2 > /dev/null
!aria2c -x 16 -s 16 -k 1M -d /content -o AquaScan-1K.zip "https://zenodo.org/records/18771165/files/AquaScan-1K.zip?download=1"

In [ ]:
!rm /content/AquaScan-1K.zip
!mv /content/AquaScan-1K.1.zip /content/AquaScan-1K.zip
!cp /content/AquaScan-1K.zip /content/drive/MyDrive/sonar_proje/
!unzip -q /content/AquaScan-1K.zip -d /content/aquascan
!find /content/aquascan -maxdepth 3 | head -20

In [ ]:
!ls /content/aquascan/*/images | head -20
!ls /content/aquascan/*/images | wc -l
!head -3 $(ls /content/aquascan/*/labels/*.txt | head -1)
!cat /content/aquascan/*/classes.txt

In [ ]:
import os, re, glob, json, datetime as dt
from collections import defaultdict

BASE = '/content/drive/MyDrive/sonar_proje'
SRC = glob.glob('/content/aquascan/*/images')[0]
LBL = SRC.replace('/images', '/labels')
files = sorted(os.listdir(SRC))

MONTHS = {m: i+1 for i, m in enumerate(
    ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC'])}

def parse_dt(name):
    n = re.sub(r'^[0-9a-f]{8}-', '', name)
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})[_-](\d{2})[._h](\d{2})[._m](\d{2})', n)
    if m:
        return dt.datetime(*map(int, m.groups()))
    m = re.match(r'(\d{10})_(\d{2})([A-Z]{3})(\d{2})_(\d{2})(\d{2})', n)
    if m:
        _, d, mon, y, H, M = m.groups()
        return dt.datetime(2000+int(y), MONTHS[mon], int(d), int(H), int(M))
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})', n)          # sadece tarih olanlar
    if m:
        return dt.datetime(*map(int, m.groups()))
    return None

groups, unparsed = defaultdict(list), []
for f in files:
    d = parse_dt(f)
    if d is None:
        unparsed.append(f)
        groups['unknown_' + f].append(f)
    else:
        groups[d.date().isoformat()].append(f)

print('toplam:', len(files), '| grup sayısı:', len(groups), '| ayrıştırılamayan:', len(unparsed))
print('ayrıştırılamayan:', unparsed)
print('küçük grup örnekleri:', [(k, len(v)) for k, v in groups.items() if len(v) <= 3][:10])

targets = {'train': .70, 'val': .15, 'test': .15}
N = len(files)
split = {k: [] for k in targets}
for g in sorted(groups, key=lambda g: (-len(groups[g]), g)):
    deficit = {k: targets[k]*N - len(split[k]) for k in targets}
    split[max(deficit, key=deficit.get)] += groups[g]

def n_obj(f):
    p = f'{LBL}/{os.path.splitext(f)[0]}.txt'
    return len(open(p).read().splitlines()) if os.path.exists(p) else 0

for k, v in split.items():
    print(k, len(v), 'görüntü,', sum(n_obj(f) for f in v), 'hedef,',
          sum(1 for f in v if n_obj(f) == 0), 'boş etiketli')

json.dump(split, open(f'{BASE}/split.json', 'w'))

In [ ]:
import shutil
ROOT = '/content/dataset_orig'
for s, fl in split.items():
    os.makedirs(f'{ROOT}/images/{s}', exist_ok=True)
    os.makedirs(f'{ROOT}/labels/{s}', exist_ok=True)
    for f in fl:
        shutil.copy(f'{SRC}/{f}', f'{ROOT}/images/{s}/{f}')
        lb = f'{LBL}/{os.path.splitext(f)[0]}.txt'
        if os.path.exists(lb):
            shutil.copy(lb, f'{ROOT}/labels/{s}/')

open(f'{ROOT}/data.yaml', 'w').write(f"""path: {ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: Human
""")

# kontrol
for s in split:
    print(s, len(os.listdir(f'{ROOT}/images/{s}')), 'görüntü,', len(os.listdir(f'{ROOT}/labels/{s}')), 'etiket')

# görüntü boyutları
from PIL import Image
import random
sample = random.Random(0).sample(os.listdir(f'{ROOT}/images/train'), 10)
print([Image.open(f'{ROOT}/images/train/{f}').size for f in sample])

In [ ]:
!cp -r /content/dataset_orig /content/drive/MyDrive/sonar_proje/dataset_orig

In [ ]:
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'

model = YOLO('yolo11s.pt')
model.train(
    data='/content/dataset_orig/data.yaml',
    epochs=100, patience=30,
    imgsz=960, batch=8,
    seed=0, deterministic=True,
    workers=2, cache='ram',
    project=f'{BASE}/runs', name='baseline_orig',
    plots=True,
)

In [ ]:
import pandas as pd, glob, os
import matplotlib.pyplot as plt

BASE = '/content/drive/MyDrive/sonar_proje'
run = sorted(glob.glob(f'{BASE}/runs/baseline_orig*'), key=os.path.getmtime)[-1]
df = pd.read_csv(f'{run}/results.csv')
df.columns = [c.strip() for c in df.columns]

best = df['metrics/mAP50-95(B)'].idxmax()
print('klasör:', run)
print('toplam epoch:', len(df), '| en iyi epoch:', int(df.loc[best, 'epoch']))
print(df.loc[[best, len(df)-1], ['epoch',
      'metrics/precision(B)', 'metrics/recall(B)',
      'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].round(4).to_string(index=False))

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for a, k in zip(ax.flat[:3], ['box_loss', 'cls_loss', 'dfl_loss']):
    a.plot(df['epoch'], df[f'train/{k}'], label='train')
    a.plot(df['epoch'], df[f'val/{k}'], label='val')
    a.set_title(k); a.set_xlabel('epoch'); a.legend()
ax.flat[3].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5')
ax.flat[3].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
ax.flat[3].set_title('val mAP'); ax.flat[3].set_xlabel('epoch'); ax.flat[3].legend()
plt.tight_layout()
plt.savefig(f'{BASE}/baseline_loss_curves.png', dpi=150)
plt.show()

In [ ]:
import glob, os, pandas as pd
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
RES = f'{BASE}/results_table.csv'
EXP = 'orijinal'     # NLM ve BM3D hatlarında 'nlm' ve 'bm3d' yazacaksın

run = sorted(glob.glob(f'{BASE}/runs/baseline_orig*'), key=os.path.getmtime)[-1]
m = YOLO(f'{run}/weights/best.pt')

rows = []
for split in ['val', 'test']:
    r = m.val(data='/content/dataset_orig/data.yaml', split=split,
              imgsz=960, batch=8, plots=True, verbose=False,
              project=f'{BASE}/runs', name=f'eval_baseline_{split}')
    P, R = r.box.mp, r.box.mr
    rows.append(dict(deney=EXP, kume=split, precision=P, recall=R,
                     f1=2*P*R/(P+R), mAP50=r.box.map50, mAP50_95=r.box.map))

new = pd.DataFrame(rows).round(4)
if os.path.exists(RES):
    new = pd.concat([pd.read_csv(RES), new]).drop_duplicates(['deney', 'kume'], keep='last')
new.to_csv(RES, index=False)
print(new.to_string(index=False))

In [ ]:
import cv2, time, os, random, numpy as np, bm3d
import matplotlib.pyplot as plt

ROOT = '/content/dataset_orig'
BASE = '/content/drive/MyDrive/sonar_proje'
files = sorted(os.listdir(f'{ROOT}/images/val'))
random.Random(1).shuffle(files)
samples = files[:3]

def load(f):
    im = cv2.imread(f'{ROOT}/images/val/{f}', cv2.IMREAD_UNCHANGED)
    if im.ndim == 3 and im.shape[2] >= 3 and np.array_equal(im[..., 0], im[..., 1]):
        im = im[..., 0]                         # gri görüntü ise tek kanala indir
    elif im.ndim == 3:
        im = im[..., :3]
    return im

def nlm(im, h):
    if im.ndim == 2:
        return cv2.fastNlMeansDenoising(im, None, h, 7, 21)
    return cv2.fastNlMeansDenoisingColored(im, None, h, h, 7, 21)

def bm3d_(im, sigma):
    x = im.astype(np.float32) / 255
    fn = bm3d.bm3d if im.ndim == 2 else bm3d.bm3d_rgb
    return (np.clip(fn(x, sigma_psd=sigma / 255), 0, 1) * 255).astype(np.uint8)

im0 = load(samples[0])
print('görüntü şekli:', im0.shape, '(2 boyut = gri)')

def crop_around_target(f, im, w=360, h=220):
    lb = open(f'{ROOT}/labels/val/{os.path.splitext(f)[0]}.txt').read().split()[1:5]
    cx, cy = float(lb[0]) * im.shape[1], float(lb[1]) * im.shape[0]
    x0 = int(np.clip(cx - w/2, 0, im.shape[1] - w)); y0 = int(np.clip(cy - h/2, 0, im.shape[0] - h))
    return im[y0:y0+h, x0:x0+w]

cols = [('Orijinal', lambda c: c)] + \
       [(f'NLM h={h}', (lambda c, h=h: nlm(c, h))) for h in (5, 10, 15)] + \
       [(f'BM3D σ={s}', (lambda c, s=s: bm3d_(c, s))) for s in (10, 20, 30)]

fig, ax = plt.subplots(len(samples), len(cols), figsize=(3.2*len(cols), 2.3*len(samples)))
for i, f in enumerate(samples):
    crop = crop_around_target(f, load(f))
    for j, (name, fn) in enumerate(cols):
        ax[i, j].imshow(fn(crop), cmap='gray' if crop.ndim == 2 else None)
        ax[i, j].axis('off')
        if i == 0: ax[i, j].set_title(name)
plt.tight_layout()
plt.savefig(f'{BASE}/denoise_examples.png', dpi=150)
plt.show()

# süre ölçümü: tam boyut tek görüntü
t = time.time(); nlm(im0, 10); t_nlm = time.time() - t
t = time.time(); bm3d_(im0, 20); t_bm = time.time() - t
print(f'NLM: {t_nlm:.1f} sn/görüntü -> 1033 görüntü ≈ {t_nlm*1033/60:.0f} dk')
print(f'BM3D: {t_bm:.1f} sn/görüntü -> 1033 görüntü ≈ {t_bm*1033/60:.0f} dk')

In [ ]:
import cv2, os, json, shutil, numpy as np
from PIL import Image
from tqdm import tqdm

BASE = '/content/drive/MyDrive/sonar_proje'
SRC = '/content/dataset_orig' if os.path.exists('/content/dataset_orig') else f'{BASE}/dataset_orig'
DST = f'{BASE}/dataset_nlm'
PARAMS = dict(method='NLM', h=10, hColor=10, templateWindowSize=7, searchWindowSize=21)
json.dump(PARAMS, open(f'{BASE}/nlm_params.json', 'w'))

def read_bgr(path):
    im = cv2.imread(path, cv2.IMREAD_COLOR)
    if im is not None and im.dtype == np.uint8 and im.ndim == 3 and im.shape[2] == 3:
        return np.ascontiguousarray(im)
    pil = Image.open(path)
    print('özel format:', os.path.basename(path), pil.mode, None if im is None else (im.dtype, im.shape))
    a = np.array(pil)
    if a.dtype == np.uint16:
        pil = Image.fromarray((a / 257).astype(np.uint8))
    elif a.dtype != np.uint8:
        pil = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    return cv2.cvtColor(np.array(pil.convert('RGB')), cv2.COLOR_RGB2BGR)

todo = []
for s in ['train', 'val', 'test']:
    os.makedirs(f'{DST}/images/{s}', exist_ok=True)
    os.makedirs(f'{DST}/labels/{s}', exist_ok=True)
    for f in sorted(os.listdir(f'{SRC}/images/{s}')):
        todo.append((s, f))
        lb = os.path.splitext(f)[0] + '.txt'
        shutil.copy(f'{SRC}/labels/{s}/{lb}', f'{DST}/labels/{s}/{lb}')

failed = []
for s, f in tqdm(todo):
    out = f'{DST}/images/{s}/{os.path.splitext(f)[0]}.png'
    if os.path.exists(out):
        continue
    try:
        im = read_bgr(f'{SRC}/images/{s}/{f}')
        den = cv2.fastNlMeansDenoisingColored(im, None, 10, 10, 7, 21)
        cv2.imwrite(out + '.tmp.png', den)
        os.replace(out + '.tmp.png', out)
    except Exception as e:
        failed.append((s, f, str(e)[:100]))
        print('HATA:', s, f, str(e)[:100])

print('başarısız:', failed)
if not failed:
    if not os.path.exists('/content/dataset_nlm'):
        shutil.copytree(DST, '/content/dataset_nlm')
    open('/content/dataset_nlm/data.yaml', 'w').write("""path: /content/dataset_nlm
train: images/train
val: images/val
test: images/test
names:
  0: Human
""")
    print('bitti:', {s: len(os.listdir(f'/content/dataset_nlm/images/{s}')) for s in ['train', 'val', 'test']})

In [ ]:
import numpy as np
c = np.load('/content/dataset_orig/labels/train.cache', allow_pickle=True).item()
print({k: v for k, v in c.items() if k in ('hash', 'results', 'msgs')})

In [ ]:
import os, json, shutil
BASE = '/content/drive/MyDrive/sonar_proje'
DST = f'{BASE}/dataset_nlm'
bad = ['9bf66ea1-Screenshot_2025-08-03_15.08.49']

for b in bad:
    for p in (f'{DST}/labels/train/{b}.txt', f'{DST}/images/train/{b}.png'):
        if os.path.exists(p):
            os.remove(p)
json.dump(bad, open(f'{BASE}/excluded_images.json', 'w'))

if os.path.exists('/content/dataset_nlm'):
    shutil.rmtree('/content/dataset_nlm')
shutil.copytree(DST, '/content/dataset_nlm')
open('/content/dataset_nlm/data.yaml', 'w').write("""path: /content/dataset_nlm
train: images/train
val: images/val
test: images/test
names:
  0: Human
""")
print({s: len(os.listdir(f'/content/dataset_nlm/images/{s}')) for s in ['train', 'val', 'test']})

In [ ]:
import glob, os, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
EXP, NAME = 'nlm', 'nlm'
DATA = '/content/dataset_nlm/data.yaml'

model = YOLO('yolo11s.pt')
model.train(data=DATA, epochs=100, patience=30, imgsz=960, batch=8,
            seed=0, deterministic=True, workers=2, cache='ram',
            project=f'{BASE}/runs', name=f'train_{NAME}', plots=True)

run = sorted(glob.glob(f'{BASE}/runs/train_{NAME}*'), key=os.path.getmtime)[-1]
m = YOLO(f'{run}/weights/best.pt')

rows = []
for split in ['val', 'test']:
    r = m.val(data=DATA, split=split, imgsz=960, batch=8, plots=True, verbose=False,
              project=f'{BASE}/runs', name=f'eval_{NAME}_{split}')
    P, R = r.box.mp, r.box.mr
    rows.append(dict(deney=EXP, kume=split, precision=P, recall=R,
                     f1=2*P*R/(P+R), mAP50=r.box.map50, mAP50_95=r.box.map))

RES = f'{BASE}/results_table.csv'
new = pd.DataFrame(rows).round(4)
if os.path.exists(RES):
    new = pd.concat([pd.read_csv(RES), new]).drop_duplicates(['deney', 'kume'], keep='last')
new.to_csv(RES, index=False)
print(new.to_string(index=False))

df = pd.read_csv(f'{run}/results.csv'); df.columns = [c.strip() for c in df.columns]
print('toplam epoch:', len(df), '| en iyi epoch:', int(df.loc[df['metrics/mAP50-95(B)'].idxmax(), 'epoch']))
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for a, k in zip(ax.flat[:3], ['box_loss', 'cls_loss', 'dfl_loss']):
    a.plot(df['epoch'], df[f'train/{k}'], label='train'); a.plot(df['epoch'], df[f'val/{k}'], label='val')
    a.set_title(k); a.set_xlabel('epoch'); a.legend()
ax.flat[3].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5')
ax.flat[3].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
ax.flat[3].set_title('val mAP'); ax.flat[3].set_xlabel('epoch'); ax.flat[3].legend()
plt.tight_layout(); plt.savefig(f'{BASE}/{NAME}_loss_curves.png', dpi=150); plt.show()

In [ ]:
import bm3d, time, cv2, numpy as np, os, random
from bm3d import BM3DProfile, BM3DStages

print('profil ayarları:', {k: v for k, v in vars(BM3DProfile()).items() if not k.startswith('_')})

ROOT = '/content/dataset_orig'
files = sorted(os.listdir(f'{ROOT}/images/val')); random.Random(1).shuffle(files)
im = cv2.imread(f'{ROOT}/images/val/{files[0]}')
y = cv2.cvtColor(im, cv2.COLOR_BGR2YCrCb)[..., 0]
part = np.ascontiguousarray(y[300:570, 600:1080]).astype(np.float32) / 255   # 480x270, tam boyutun 1/16'sı
sig = 20 / 255

def run(name, **kw):
    t = time.time(); out = bm3d.bm3d(part, sigma_psd=sig, **kw); d = time.time() - t
    print(f'{name}: {d:.1f} sn (parça) -> tam boyut ≈ {d*16:.0f} sn -> 1033 görüntü ≈ {d*16*1033/3600:.1f} saat')
    return out

run('1) tam (HT + Wiener), sadece Y')
run('2) sadece HT aşaması', stage_arg=BM3DStages.HARD_THRESHOLDING)

p = BM3DProfile()
for attr, val in [('step_ht', 6), ('search_window_ht', 19)]:
    if hasattr(p, attr):
        setattr(p, attr, val)
    else:
        print('profilde yok:', attr)
run('3) HT + büyük adım + dar arama', profile=p, stage_arg=BM3DStages.HARD_THRESHOLDING)

In [ ]:
import bm3d, cv2, os, random, numpy as np
import matplotlib.pyplot as plt
from bm3d import BM3DProfile, BM3DStages

BASE = '/content/drive/MyDrive/sonar_proje'
ROOT = '/content/dataset_orig'

def make_profile():
    p = BM3DProfile(); p.step_ht = 6; p.search_window_ht = 19
    return p

def bm3d_fast(bgr, sigma=20):
    ycc = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)
    y = ycc[..., 0].astype(np.float32) / 255
    out = bm3d.bm3d(y, sigma_psd=sigma/255, profile=make_profile(), stage_arg=BM3DStages.HARD_THRESHOLDING)
    ycc[..., 0] = np.clip(out * 255 + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)

files = sorted(os.listdir(f'{ROOT}/images/val')); random.Random(1).shuffle(files)
samples = files[:3]
W, H = 360, 220
fig, ax = plt.subplots(3, 4, figsize=(15, 8.5))
for i, f in enumerate(samples):
    stem = os.path.splitext(f)[0]
    orig = cv2.imread(f'{ROOT}/images/val/{f}')
    lb = open(f'{ROOT}/labels/val/{stem}.txt').read().split()[1:5]
    cx, cy = float(lb[0]) * orig.shape[1], float(lb[1]) * orig.shape[0]
    x0 = int(np.clip(cx - W/2, 0, orig.shape[1] - W)); y0 = int(np.clip(cy - H/2, 0, orig.shape[0] - H))
    c = np.ascontiguousarray(orig[y0:y0+H, x0:x0+W])
    nlm_path = f'/content/dataset_nlm/images/val/{stem}.png'
    nlm = cv2.imread(nlm_path)[y0:y0+H, x0:x0+W] if os.path.exists(nlm_path) else c
    for j, (name, im) in enumerate([('Orijinal', c), ('NLM h=10', nlm),
                                    ('BM3D hızlı σ=20', bm3d_fast(c, 20)), ('BM3D hızlı σ=30', bm3d_fast(c, 30))]):
        ax[i, j].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax[i, j].axis('off')
        if i == 0: ax[i, j].set_title(name)
plt.tight_layout(); plt.savefig(f'{BASE}/denoise_examples_bm3d_fast.png', dpi=150); plt.show()

In [ ]:
import json, shutil
from tqdm import tqdm
from PIL import Image

SRC = ROOT
DST = f'{BASE}/dataset_bm3d'
SIGMA = 20
json.dump(dict(method='BM3D (bm3d paketi, hızlı yapılandırma)', sigma=SIGMA, kanal='Y (YCrCb), Cr/Cb değişmedi',
               stage='HARD_THRESHOLDING', step_ht=6, search_window_ht=19),
          open(f'{BASE}/bm3d_params.json', 'w'), ensure_ascii=False)

bad = set(json.load(open(f'{BASE}/excluded_images.json'))) if os.path.exists(f'{BASE}/excluded_images.json') else set()

def read_bgr(path):
    im = cv2.imread(path, cv2.IMREAD_COLOR)
    if im is not None and im.dtype == np.uint8 and im.ndim == 3 and im.shape[2] == 3:
        return np.ascontiguousarray(im)
    a = np.array(Image.open(path))
    if a.dtype == np.uint16: a = (a / 257).astype(np.uint8)
    return cv2.cvtColor(np.array(Image.fromarray(a.astype(np.uint8)).convert('RGB')), cv2.COLOR_RGB2BGR)

todo = []
for s in ['train', 'val', 'test']:
    os.makedirs(f'{DST}/images/{s}', exist_ok=True); os.makedirs(f'{DST}/labels/{s}', exist_ok=True)
    for f in sorted(os.listdir(f'{SRC}/images/{s}')):
        stem = os.path.splitext(f)[0]
        if stem in bad: continue
        todo.append((s, f))
        shutil.copy(f'{SRC}/labels/{s}/{stem}.txt', f'{DST}/labels/{s}/{stem}.txt')

failed = []
for s, f in tqdm(todo):
    out = f'{DST}/images/{s}/{os.path.splitext(f)[0]}.png'
    if os.path.exists(out): continue
    try:
        cv2.imwrite(out + '.tmp.png', bm3d_fast(read_bgr(f'{SRC}/images/{s}/{f}'), SIGMA))
        os.replace(out + '.tmp.png', out)
    except Exception as e:
        failed.append((s, f, str(e)[:100])); print('HATA:', s, f, str(e)[:100])

print('başarısız:', failed)
if not failed:
    if os.path.exists('/content/dataset_bm3d'): shutil.rmtree('/content/dataset_bm3d')
    shutil.copytree(DST, '/content/dataset_bm3d')
    open('/content/dataset_bm3d/data.yaml', 'w').write("""path: /content/dataset_bm3d
train: images/train
val: images/val
test: images/test
names:
  0: Human
""")
    print('bitti:', {s: len(os.listdir(f'/content/dataset_bm3d/images/{s}')) for s in ['train', 'val', 'test']})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q bm3d

import os, json, shutil, cv2, numpy as np, bm3d
from PIL import Image
from tqdm import tqdm
from bm3d import BM3DProfile, BM3DStages

BASE = '/content/drive/MyDrive/sonar_proje'
ROOT = '/content/dataset_orig'
if not os.path.exists(ROOT):
    assert os.path.exists(f'{BASE}/dataset_orig'), 'Drive yedeği yok -> bana yaz'
    shutil.copytree(f'{BASE}/dataset_orig', ROOT)

def make_profile():
    p = BM3DProfile(); p.step_ht = 6; p.search_window_ht = 19
    return p

def bm3d_fast(bgr, sigma=20):
    ycc = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)
    y = ycc[..., 0].astype(np.float32) / 255
    out = bm3d.bm3d(y, sigma_psd=sigma/255, profile=make_profile(), stage_arg=BM3DStages.HARD_THRESHOLDING)
    ycc[..., 0] = np.clip(out * 255 + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)

def read_bgr(path):
    im = cv2.imread(path, cv2.IMREAD_COLOR)
    if im is not None and im.dtype == np.uint8 and im.ndim == 3 and im.shape[2] == 3:
        return np.ascontiguousarray(im)
    a = np.array(Image.open(path))
    if a.dtype == np.uint16: a = (a / 257).astype(np.uint8)
    return cv2.cvtColor(np.array(Image.fromarray(a.astype(np.uint8)).convert('RGB')), cv2.COLOR_RGB2BGR)

SRC, DST, SIGMA = ROOT, f'{BASE}/dataset_bm3d', 20
json.dump(dict(method='BM3D (bm3d paketi, hızlı yapılandırma)', sigma=SIGMA, kanal='Y (YCrCb), Cr/Cb değişmedi',
               stage='HARD_THRESHOLDING', step_ht=6, search_window_ht=19),
          open(f'{BASE}/bm3d_params.json', 'w'), ensure_ascii=False)
bad = set(json.load(open(f'{BASE}/excluded_images.json'))) if os.path.exists(f'{BASE}/excluded_images.json') else set()

todo = []
for s in ['train', 'val', 'test']:
    os.makedirs(f'{DST}/images/{s}', exist_ok=True); os.makedirs(f'{DST}/labels/{s}', exist_ok=True)
    for f in sorted(os.listdir(f'{SRC}/images/{s}')):
        stem = os.path.splitext(f)[0]
        if stem in bad: continue
        todo.append((s, f))
        shutil.copy(f'{SRC}/labels/{s}/{stem}.txt', f'{DST}/labels/{s}/{stem}.txt')

kalan = sum(not os.path.exists(f'{DST}/images/{s}/{os.path.splitext(f)[0]}.png') for s, f in todo)
print(f'toplam {len(todo)} | yapılmış {len(todo)-kalan} | kalan {kalan}')

failed = []
for s, f in tqdm(todo):
    out = f'{DST}/images/{s}/{os.path.splitext(f)[0]}.png'
    if os.path.exists(out): continue
    try:
        cv2.imwrite(out + '.tmp.png', bm3d_fast(read_bgr(f'{SRC}/images/{s}/{f}'), SIGMA))
        os.replace(out + '.tmp.png', out)
    except Exception as e:
        failed.append((s, f, str(e)[:100])); print('HATA:', s, f, str(e)[:100])

print('başarısız:', failed)
if not failed:
    if os.path.exists('/content/dataset_bm3d'): shutil.rmtree('/content/dataset_bm3d')
    shutil.copytree(DST, '/content/dataset_bm3d')
    open('/content/dataset_bm3d/data.yaml', 'w').write("""path: /content/dataset_bm3d
train: images/train
val: images/val
test: images/test
names:
  0: Human
""")
    print('bitti:', {s: len(os.listdir(f'/content/dataset_bm3d/images/{s}')) for s in ['train', 'val', 'test']})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/sonar_proje'
print(sorted(os.listdir(D)))
for s in ['train', 'val', 'test']:
    print(s, 'resim:', len(os.listdir(f'{D}/dataset_bm3d/images/{s}')), '| etiket:', len(os.listdir(f'{D}/dataset_bm3d/labels/{s}')))
drive.flush_and_unmount()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

import torch, os, glob, shutil, pandas as pd
import matplotlib.pyplot as plt
assert torch.cuda.is_available(), 'GPU yok -> Çalışma zamanı türünü T4 GPU yap ve tekrar çalıştır'
print('GPU:', torch.cuda.get_device_name(0))

from ultralytics import YOLO
BASE = '/content/drive/MyDrive/sonar_proje'
EXP, NAME = 'bm3d', 'bm3d'

if not os.path.exists('/content/dataset_bm3d'):
    shutil.copytree(f'{BASE}/dataset_bm3d', '/content/dataset_bm3d')
open('/content/dataset_bm3d/data.yaml', 'w').write("""path: /content/dataset_bm3d
train: images/train
val: images/val
test: images/test
names:
  0: Human
""")
DATA = '/content/dataset_bm3d/data.yaml'
print({s: len(os.listdir(f'/content/dataset_bm3d/images/{s}')) for s in ['train', 'val', 'test']})

model = YOLO('yolo11s.pt')
model.train(data=DATA, epochs=100, patience=30, imgsz=960, batch=8,
            seed=0, deterministic=True, workers=2, cache='ram',
            project=f'{BASE}/runs', name=f'train_{NAME}', plots=True)

run = sorted(glob.glob(f'{BASE}/runs/train_{NAME}*'), key=os.path.getmtime)[-1]
m = YOLO(f'{run}/weights/best.pt')
rows = []
for split in ['val', 'test']:
    r = m.val(data=DATA, split=split, imgsz=960, batch=8, plots=True, verbose=False,
              project=f'{BASE}/runs', name=f'eval_{NAME}_{split}')
    P, R = r.box.mp, r.box.mr
    rows.append(dict(deney=EXP, kume=split, precision=P, recall=R,
                     f1=2*P*R/(P+R), mAP50=r.box.map50, mAP50_95=r.box.map))

RES = f'{BASE}/results_table.csv'
new = pd.DataFrame(rows).round(4)
if os.path.exists(RES):
    new = pd.concat([pd.read_csv(RES), new]).drop_duplicates(['deney', 'kume'], keep='last')
new.to_csv(RES, index=False)
print(new.to_string(index=False))

df = pd.read_csv(f'{run}/results.csv'); df.columns = [c.strip() for c in df.columns]
print('toplam epoch:', len(df), '| en iyi epoch:', int(df.loc[df['metrics/mAP50-95(B)'].idxmax(), 'epoch']))
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for a, k in zip(ax.flat[:3], ['box_loss', 'cls_loss', 'dfl_loss']):
    a.plot(df['epoch'], df[f'train/{k}'], label='train'); a.plot(df['epoch'], df[f'val/{k}'], label='val')
    a.set_title(k); a.set_xlabel('epoch'); a.legend()
    if k == 'cls_loss': a.set_ylim(0, 5)
ax.flat[3].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5')
ax.flat[3].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
ax.flat[3].set_title('val mAP'); ax.flat[3].set_xlabel('epoch'); ax.flat[3].legend()
plt.tight_layout(); plt.savefig(f'{BASE}/{NAME}_loss_curves.png', dpi=150); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

import os, glob, json, cv2, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
def latest(p): return sorted(glob.glob(p), key=os.path.getmtime)[-1]
MODELS = {
    'Orijinal': (f"{latest(f'{BASE}/runs/baseline_orig*')}/weights/best.pt", 'dataset_orig'),
    'NLM':      (f"{latest(f'{BASE}/runs/train_nlm*')}/weights/best.pt",     'dataset_nlm'),
    'BM3D':     (f"{latest(f'{BASE}/runs/train_bm3d*')}/weights/best.pt",    'dataset_bm3d'),
}
test_files = json.load(open(f'{BASE}/split.json'))['test']

def img_path(name, f):
    root = MODELS[name][1]
    return f'{BASE}/{root}/images/test/' + (f if root == 'dataset_orig' else os.path.splitext(f)[0] + '.png')

gts = []   # (dosya, normalize xyxy, alan)
for f in test_files:
    for line in open(f'{BASE}/dataset_orig/labels/test/{os.path.splitext(f)[0]}.txt').read().splitlines():
        c, cx, cy, w, h = map(float, line.split()[:5])
        gts.append((f, np.array([cx-w/2, cy-h/2, cx+w/2, cy+h/2]), w*h))
small = list(np.argsort([g[2] for g in gts])[:max(1, len(gts)//4)])    # en küçük %25
print('toplam hedef:', len(gts), '| küçük hedef sayısı:', len(small))

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0

preds = {}
for name, (wp, root) in MODELS.items():
    print('tahmin yapılıyor:', name)
    m = YOLO(wp); preds[name] = {}
    paths = [img_path(name, f) for f in test_files]
    for f, r in zip(test_files, m.predict(paths, imgsz=960, conf=0.25, verbose=False, stream=True)):
        preds[name][f] = [(b.xyxyn[0].cpu().numpy(), float(b.conf[0])) for b in r.boxes]

def hit(name, g):
    return any(iou(g[1], p) >= 0.5 for p, _ in preds[name][g[0]])

print(f"\n{'Model':10} {'Küçük hedef':>14} {'Tüm hedef':>12} {'Yanlış alarm':>13}")
for name in MODELS:
    hs = [hit(name, g) for g in gts]
    fp = sum(1 for f in test_files for p, c in preds[name][f]
             if not any(iou(g[1], p) >= 0.5 for g in gts if g[0] == f))
    print(f"{name:10} {sum(hs[i] for i in small):>8}/{len(small):<5} {sum(hs):>6}/{len(gts):<5} {fp:>13}")

flags = {i: [hit(n, gts[i]) for n in MODELS] for i in small}
disagree = [i for i in small if len(set(flags[i])) > 1]          # modellerin ayrıştığı hedefler öne
chosen = (disagree + [i for i in small if i not in disagree])[:6]

def to_px(b, W, H, x0, y0):
    return [int(v) for v in (b * np.array([W, H, W, H]) - np.array([x0, y0, x0, y0]))]

CW, CH = 480, 300
fig, ax = plt.subplots(len(chosen), len(MODELS), figsize=(5*len(MODELS), 3.3*len(chosen)))
ax = np.atleast_2d(ax)
for ri, gi in enumerate(chosen):
    f, box, _ = gts[gi]
    for ci, name in enumerate(MODELS):
        im = cv2.imread(img_path(name, f)); H, W = im.shape[:2]
        cx, cy = (box[0]+box[2])/2*W, (box[1]+box[3])/2*H
        x0 = int(np.clip(cx - CW/2, 0, max(W-CW, 0))); y0 = int(np.clip(cy - CH/2, 0, max(H-CH, 0)))
        crop = im[y0:y0+CH, x0:x0+CW].copy()
        g = to_px(box, W, H, x0, y0)
        cv2.rectangle(crop, (g[0], g[1]), (g[2], g[3]), (0, 255, 0), 2)          # yeşil = gerçek
        for p, conf in preds[name][f]:
            q = to_px(p, W, H, x0, y0)
            cv2.rectangle(crop, (q[0], q[1]), (q[2], q[3]), (0, 0, 255), 2)       # kırmızı = tahmin
            cv2.putText(crop, f'{conf:.2f}', (q[0], max(q[1]-4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        ax[ri, ci].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax[ri, ci].axis('off')
        if ri == 0: ax[ri, ci].set_title(name)
fig.suptitle('Yeşil: gerçek kutu | Kırmızı: tahmin (güven ≥ 0.25)')
plt.tight_layout(); plt.savefig(f'{BASE}/kucuk_hedef_karsilastirma.png', dpi=130); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

import os, gc, shutil, torch, pandas as pd
from ultralytics import YOLO
assert torch.cuda.is_available(), 'GPU yok -> Çalışma zamanı türünü T4 GPU yap ve tekrar çalıştır'
gpu = torch.cuda.get_device_name(0); print('GPU:', gpu)
if 'T4' not in gpu: print('UYARI: seed 0 çalışmaları T4 idi, tutarlılık için T4 tercih et')

BASE = '/content/drive/MyDrive/sonar_proje'
SEEDS = [1, 2]                       # sadece 1 seed ile başlamak istersen [1] yaz
EXPS = {'orijinal': 'dataset_orig', 'nlm': 'dataset_nlm', 'bm3d': 'dataset_bm3d'}
AUTO_DISCONNECT = True               # iş bitince oturumu kapatır (birim harcamasın)
OUT = f'{BASE}/runs_seeds'; os.makedirs(OUT, exist_ok=True)
CSV = f'{BASE}/results_seeds.csv'

DATA = {}
for exp, d in EXPS.items():
    if not os.path.exists(f'/content/{d}'):
        shutil.copytree(f'{BASE}/{d}', f'/content/{d}')
    open(f'/content/{d}/data.yaml', 'w').write(
        f"path: /content/{d}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n")
    DATA[exp] = f'/content/{d}/data.yaml'
    print(d, {s: len(os.listdir(f'/content/{d}/images/{s}')) for s in ['train', 'val', 'test']})

def log_rows(rows):
    new = pd.DataFrame(rows).round(4)
    if os.path.exists(CSV):
        new = pd.concat([pd.read_csv(CSV), new])
    new.drop_duplicates(['deney', 'seed', 'kume'], keep='last').to_csv(CSV, index=False)

def run_one(exp, seed):
    name = f'seed{seed}_{exp}'
    done = f'{OUT}/{name}.done'
    if os.path.exists(done):
        print('atlandı (bitmiş):', name); return
    rdir = f'{OUT}/{name}'
    best, last = f'{rdir}/weights/best.pt', f'{rdir}/weights/last.pt'
    print(f'\n=== {name} başlıyor ===')
    if os.path.exists(last):
        try:
            YOLO(last).train(resume=True)
        except Exception as e:
            print('devam edilemedi (muhtemelen zaten bitmiş):', str(e)[:150])
    else:
        YOLO('yolo11s.pt').train(data=DATA[exp], epochs=100, patience=30, imgsz=960, batch=8,
                                 seed=seed, deterministic=True, workers=2, cache='ram',
                                 project=OUT, name=name, exist_ok=True, plots=False)
    m = YOLO(best); rows = []
    for split in ['val', 'test']:
        r = m.val(data=DATA[exp], split=split, imgsz=960, batch=8, verbose=False, plots=False,
                  project=OUT, name=f'eval_{name}_{split}', exist_ok=True)
        P, R = r.box.mp, r.box.mr
        rows.append(dict(deney=exp, seed=seed, kume=split, precision=P, recall=R,
                         f1=2*P*R/(P+R) if P+R > 0 else 0, mAP50=r.box.map50, mAP50_95=r.box.map))
    log_rows(rows); open(done, 'w').write('ok')
    del m; gc.collect(); torch.cuda.empty_cache()

for seed in SEEDS:
    for exp in EXPS:
        run_one(exp, seed)

# --- seed 0 ile birleştir, ortalama ± std ---
s0 = pd.read_csv(f'{BASE}/results_table.csv'); s0['seed'] = 0
allr = pd.concat([s0, pd.read_csv(CSV)]).drop_duplicates(['deney', 'seed', 'kume'], keep='last')
allr.to_csv(f'{BASE}/results_all_seeds.csv', index=False)
cols = ['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']
for kume in ['test', 'val']:
    print(f'\n== {kume}: ortalama ± std ==')
    for exp in EXPS:
        d = allr[(allr.deney == exp) & (allr.kume == kume)]
        print(exp.ljust(9), f'n={len(d)}', ' | '.join(f'{c} {d[c].mean():.3f}±{d[c].std():.3f}' for c in cols))

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil, datetime

BASE = '/content/drive/MyDrive/sonar_proje'
INCLUDE_DATASETS = False   # True: dataset_nlm ve dataset_bm3d de kopyalanır (büyük, uzun sürer)
MAKE_ZIP = True            # indirmesi kolay tek bir zip dosyası üretir

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
DST = f'/content/drive/MyDrive/sonar_proje_yedek_{stamp}'
os.makedirs(DST, exist_ok=True)

# 1) küçük ve önemli dosyalar: sonuç tabloları, parametreler, bölme, grafikler
n = 0
for pat in ('*.csv', '*.json', '*.png', '*.jpg', '*.pdf'):
    for f in glob.glob(f'{BASE}/{pat}'):
        shutil.copy2(f, DST); n += 1
print('küçük dosya sayısı:', n)

# 2) eğitim çıktıları (ağırlıklar dahil) ve isteğe bağlı veri setleri
subs = ['runs', 'runs_seeds'] + (['dataset_nlm', 'dataset_bm3d'] if INCLUDE_DATASETS else [])
for sub in subs:
    if os.path.exists(f'{BASE}/{sub}'):
        print('kopyalanıyor:', sub)
        shutil.copytree(f'{BASE}/{sub}', f'{DST}/{sub}', dirs_exist_ok=True)
    else:
        print('bulunamadı (atlandı):', sub)

# 3) ortam bilgisi ve açıklama dosyası
os.system(f'pip freeze > "{DST}/requirements_colab.txt"')
open(f'{DST}/OKU_BENI.txt', 'w', encoding='utf-8').write("""Yan taramalı sonar projesi - yedek
Veri seti: AquaScan-1K (Zenodo kaydı 18771165), 1033 görüntü, tek sınıf (insan).
Bölme: split.json (aynı güne ait görüntüler aynı kümede). Bozuk görüntü: excluded_images.json.
Model: YOLO11s (yolo11s.pt), imgsz=960, batch=8, epochs=100, patience=30, deterministic=True, Ultralytics varsayılan ayarlar.
Seed: 0 (runs/), 1 ve 2 (runs_seeds/).
NLM: cv2.fastNlMeansDenoisingColored, h=10, hColor=10, template=7, search=21 (nlm_params.json).
BM3D (hızlandırılmış): bm3d paketi, YCrCb Y kanalı, sigma=20, sadece HARD_THRESHOLDING, step_ht=6, search_window_ht=19 (bm3d_params.json).
Sonuçlar: results_table.csv (seed 0), results_seeds.csv (seed 1-2), results_all_seeds.csv (hepsi).
Veri setleri (dataset_orig/nlm/bm3d) yedeğe dahil değilse sonar_proje klasöründe durur; orijinali Zenodo'dan tekrar indirilebilir.
""")

# 4) kontrol
def size_mb(p):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs) / 1e6
print('\nYedek klasörü:', DST, f'({size_mb(DST):.0f} MB)')
print(sorted(os.listdir(DST)))
if MAKE_ZIP:
    z = shutil.make_archive(DST, 'zip', DST)
    print('Zip:', z, f'({os.path.getsize(z)/1e6:.0f} MB)')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install ultralytics

In [ ]:
import glob
BASE = '/content/drive/MyDrive/sonar_proje'
for p in sorted(glob.glob(f'{BASE}/runs*/**/weights/best.pt', recursive=True)):
    print(p)

In [ ]:
import glob, re, os
import numpy as np
from ultralytics import YOLO

BASE   = '/content/drive/MyDrive/sonar_proje'
W_ORIG = '/content/drive/MyDrive/sonar_proje/runs/baseline_orig/weights/best.pt'
W_NLM  = '/content/drive/MyDrive/sonar_proje/runs/train_nlm/weights/best.pt'
W_BM3D = '/content/drive/MyDrive/sonar_proje/runs/train_bm3d/weights/best.pt'

W     = {'orijinal': W_ORIG, 'nlm': W_NLM, 'bm3d': W_BM3D}
DS    = {'orijinal': 'dataset_orig', 'nlm': 'dataset_nlm', 'bm3d': 'dataset_bm3d'}
KENDI = {'orijinal': 0.300, 'nlm': 0.368, 'bm3d': 0.422}   # F1 grafiklerinden

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0

def read_gt(img_path, w, h):
    lbl = re.sub(r'/images/', '/labels/', img_path).rsplit('.', 1)[0] + '.txt'
    out = []
    if os.path.exists(lbl):
        for line in open(lbl):
            p = line.split()
            if len(p) >= 5:
                c, x, y, bw, bh = int(p[0]), *map(float, p[1:5])
                out.append((c, [(x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h]))
    return out

def tahminleri_al(weights, ds):
    model, out = YOLO(weights), []
    paths = sorted(glob.glob(f'{BASE}/{ds}/images/test/*.*'))
    for r in model.predict(paths, conf=0.05, imgsz=960, device='cpu',
                           verbose=False, stream=True):
        h, w = r.orig_shape
        out.append((read_gt(r.path, w, h), r.boxes.xyxy.cpu().numpy(),
                    r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)))
    return out

veri = {ad: tahminleri_al(W[ad], DS[ad]) for ad in W}

# küçük hedef = test'teki en küçük %25 kutu (piksel alanı, orijinal etiketlerden)
alanlar = [(gb[2]-gb[0])*(gb[3]-gb[1]) for gts, *_ in veri['orijinal'] for _, gb in gts]
KUCUK = np.percentile(alanlar, 25)

def say(v, conf):
    tp = fp = n = ktp = kn = 0
    for gts, boxes, confs, cls in v:
        k = confs >= conf
        boxes, confs, cls = boxes[k], confs[k], cls[k]
        used = set()
        for i in np.argsort(-confs):
            best, bj = 0, None
            for j, (gc, gb) in enumerate(gts):
                if j in used or gc != cls[i]: continue
                s = iou(boxes[i], gb)
                if s > best: best, bj = s, j
            if bj is not None and best >= 0.5: used.add(bj); tp += 1
            else: fp += 1
        for j, (gc, gb) in enumerate(gts):
            n += 1
            if (gb[2]-gb[0])*(gb[3]-gb[1]) <= KUCUK:
                kn += 1; ktp += (j in used)
    return ktp, kn, tp, n, fp

for baslik, esik in [('KENDİ F1-max eşiği', KENDI),
                     ('HEPSİ 0.25', {a: 0.25 for a in W}),
                     ('HEPSİ 0.42', {a: 0.42 for a in W})]:
    print(f'\n=== {baslik} ===')
    for ad in W:
        ktp, kn, tp, n, fp = say(veri[ad], esik[ad])
        print(f'{ad:9s} eşik={esik[ad]:.3f} | küçük hedef {ktp}/{kn} | tüm hedef {tp}/{n} | yanlış alarm {fp}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics bm3d

In [ ]:
import os, json, time, shutil, re, cv2, numpy as np
from PIL import Image

BASE = '/content/drive/MyDrive/sonar_proje'
SRC  = f'{BASE}/dataset_orig'
H_LISTESI = [5, 15]
UZANTI = {'.png', '.jpg', '.jpeg', '.bmp'}

# daha önce dışlanan bozuk görüntüler (varsa)
dislanan = set()
yol = f'{BASE}/excluded_images.json'
if os.path.exists(yol):
    d = json.load(open(yol))
    parcalar = d if isinstance(d, list) else [x for v in (d.values() if isinstance(d, dict) else [])
                                              for x in (v if isinstance(v, list) else [v])]
    for x in parcalar:
        ad = os.path.basename(str(x))
        dislanan |= {ad, os.path.splitext(ad)[0]}
print('Dışlanan:', dislanan)

def read_bgr(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None or img.ndim != 3:
        img = cv2.cvtColor(np.array(Image.open(path).convert('RGB')), cv2.COLOR_RGB2BGR)
    return img

for h in H_LISTESI:
    DST = f'{BASE}/dataset_nlm_h{h}'
    params = dict(h=h, hColor=h, templateWindowSize=7, searchWindowSize=21)
    json.dump(params, open(f'{BASE}/nlm_params_h{h}.json', 'w'), indent=2)
    hatalar, islenen, t0 = [], 0, time.time()

    isler = []
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{DST}/images/{split}', exist_ok=True)
        os.makedirs(f'{DST}/labels/{split}', exist_ok=True)
        for f in sorted(os.listdir(f'{SRC}/images/{split}')):
            ad, uz = os.path.splitext(f)
            if uz.lower() in UZANTI and f not in dislanan and ad not in dislanan:
                isler.append((split, f, ad))
    print(f'\n=== NLM h={h} | toplam {len(isler)} görüntü ===')

    for n, (split, f, ad) in enumerate(isler, 1):
        cikis = f'{DST}/images/{split}/{ad}.png'
        # etiketi kopyala (filtre piksel yerini değiştirmediği için aynen)
        lbl = f'{SRC}/labels/{split}/{ad}.txt'
        if os.path.exists(lbl):
            shutil.copy(lbl, f'{DST}/labels/{split}/{ad}.txt')
        if os.path.exists(cikis):          # daha önce yapıldıysa atla (devam edebilmek için)
            continue
        try:
            img = read_bgr(f'{SRC}/images/{split}/{f}')
            out = cv2.fastNlMeansDenoisingColored(img, None, h, h, 7, 21)
            gecici = cikis + '.tmp.png'
            cv2.imwrite(gecici, out)
            os.replace(gecici, cikis)      # yarım dosya kalmasın
            islenen += 1
        except Exception as e:
            hatalar.append((f, str(e)[:80]))
        if n % 25 == 0:
            gecen = time.time() - t0
            print(f'{n}/{len(isler)} | geçen {gecen/60:.1f} dk | hata {len(hatalar)}')

    # data.yaml: dataset_nlm'dekini kopyala, sadece path satırını değiştir
    ornek = f'{BASE}/dataset_nlm/data.yaml'
    if os.path.exists(ornek):
        txt = re.sub(r'^path:.*$', f'path: {DST}', open(ornek).read(), flags=re.M)
    else:
        txt = f'path: {DST}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n'
    open(f'{DST}/data.yaml', 'w').write(txt)
    json.dump(hatalar, open(f'{DST}/hatalar.json', 'w'), indent=2)
    print(f'h={h} bitti | bu çalıştırmada işlenen {islenen} | hata {len(hatalar)} | süre {(time.time()-t0)/60:.1f} dk')
    if hatalar: print('Hatalı dosyalar:', hatalar)

In [ ]:
import glob
BASE = '/content/drive/MyDrive/sonar_proje'
for h in [5, 15]:
    for s in ['train', 'val', 'test']:
        dosyalar = glob.glob(f'{BASE}/dataset_nlm_h{h}/images/{s}/*.png')
        temiz = [f for f in dosyalar if not f.endswith('.tmp.png')]
        print(f'h={h} {s:5s}: {len(temiz)} görüntü | yarım (tmp): {len(dosyalar)-len(temiz)}')

In [ ]:
import os, shutil, re
import pandas as pd
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
OUT  = f'{BASE}/runs_tarama'
os.makedirs(OUT, exist_ok=True)
DENEYLER = ['nlm_h5', 'nlm_h15']      # dataset_<ad> klasörleri, sırayla eğitilir
SEED = 0

def hazirla(ad):
    """Veri setini hızlı yerel diske kopyala, data.yaml'ın path'ini düzelt."""
    src, dst = f'{BASE}/dataset_{ad}', f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    txt = re.sub(r'^path:.*$', f'path: {dst}', open(f'{src}/data.yaml').read(), flags=re.M)
    open(f'{dst}/data.yaml', 'w').write(txt)
    return f'{dst}/data.yaml'

def egit(ad):
    name = f'seed{SEED}_{ad}'
    yaml_yol = hazirla(ad)
    run = f'{OUT}/{name}'
    bitti = f'{run}/.done'
    if not os.path.exists(bitti):
        last = f'{run}/weights/last.pt'
        if os.path.exists(last):                       # yarım kaldıysa kaldığı yerden devam
            YOLO(last).train(resume=True)
        else:
            YOLO('yolo11s.pt').train(data=yaml_yol, epochs=100, patience=30, imgsz=960,
                                     batch=8, seed=SEED, deterministic=True, workers=2,
                                     cache='ram', project=OUT, name=name, exist_ok=True)
        open(bitti, 'w').write('ok')
    # sadece VAL ölçümü
    m = YOLO(f'{run}/weights/best.pt')
    r = m.val(data=yaml_yol, split='val', imgsz=960, batch=8, plots=False, verbose=False)
    return dict(deney=ad, seed=SEED, P=r.box.mp, R=r.box.mr,
                mAP50=r.box.map50, mAP50_95=r.box.map)

csv = f'{BASE}/results_tarama_val.csv'
tablo = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
for ad in DENEYLER:
    s = egit(ad)
    print(s)
    tablo = pd.concat([tablo[tablo.get('deney', pd.Series(dtype=str)) != ad], pd.DataFrame([s])])
    tablo.to_csv(csv, index=False)
print(tablo)

In [ ]:
import pandas as pd
BASE = '/content/drive/MyDrive/sonar_proje'
print(pd.read_csv(f'{BASE}/results_tarama_val.csv'))
for h in [5, 15]:
    d = pd.read_csv(f'{BASE}/runs_tarama/seed0_nlm_h{h}/results.csv')
    d.columns = d.columns.str.strip()
    m = d['metrics/mAP50-95(B)']
    print(f'h={h}: epoch sayısı {len(d)} | en iyi epoch {m.idxmax()+1} | en iyi val mAP50-95 {m.max():.3f}')
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
import os, shutil, re
import pandas as pd
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
OUT  = f'{BASE}/runs_tarama'
os.makedirs(OUT, exist_ok=True)
DENEYLER = ['nlm_h5', 'nlm_h15']
SEEDLER  = [1, 2]

def hazirla(ad):
    src, dst = f'{BASE}/dataset_{ad}', f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    txt = re.sub(r'^path:.*$', f'path: {dst}', open(f'{src}/data.yaml').read(), flags=re.M)
    open(f'{dst}/data.yaml', 'w').write(txt)
    return f'{dst}/data.yaml'

def egit(ad, seed):
    name = f'seed{seed}_{ad}'
    yaml_yol = hazirla(ad)
    run = f'{OUT}/{name}'
    bitti = f'{run}/.done'
    if not os.path.exists(bitti):
        last = f'{run}/weights/last.pt'
        if os.path.exists(last):
            YOLO(last).train(resume=True)
        else:
            YOLO('yolo11s.pt').train(data=yaml_yol, epochs=100, patience=30, imgsz=960,
                                     batch=8, seed=seed, deterministic=True, workers=2,
                                     cache='ram', project=OUT, name=name, exist_ok=True)
        open(bitti, 'w').write('ok')
    m = YOLO(f'{run}/weights/best.pt')
    r = m.val(data=yaml_yol, split='val', imgsz=960, batch=8, plots=False, verbose=False)
    return dict(deney=ad, seed=seed, P=r.box.mp, R=r.box.mr,
                mAP50=r.box.map50, mAP50_95=r.box.map)

csv = f'{BASE}/results_tarama_val.csv'
tablo = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame(
    columns=['deney', 'seed', 'P', 'R', 'mAP50', 'mAP50_95'])
for seed in SEEDLER:
    for ad in DENEYLER:
        s = egit(ad, seed)
        print(s)
        tablo = pd.concat([tablo[~((tablo.deney == ad) & (tablo.seed == seed))],
                           pd.DataFrame([s])])
        tablo.to_csv(csv, index=False)
print(tablo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

In [ ]:
import os, shutil, re
import pandas as pd
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
OUT  = f'{BASE}/runs_tarama'
os.makedirs(OUT, exist_ok=True)
DENEYLER = ['bm3d_s10']
SEEDLER  = [0, 1, 2]

def hazirla(ad):
    src, dst = f'{BASE}/dataset_{ad}', f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    txt = re.sub(r'^path:.*$', f'path: {dst}', open(f'{src}/data.yaml').read(), flags=re.M)
    open(f'{dst}/data.yaml', 'w').write(txt)
    return f'{dst}/data.yaml'

def egit(ad, seed):
    name = f'seed{seed}_{ad}'
    yaml_yol = hazirla(ad)
    run = f'{OUT}/{name}'
    bitti = f'{run}/.done'
    if not os.path.exists(bitti):
        last = f'{run}/weights/last.pt'
        if os.path.exists(last):
            YOLO(last).train(resume=True)
        else:
            YOLO('yolo11s.pt').train(data=yaml_yol, epochs=100, patience=30, imgsz=960,
                                     batch=8, seed=seed, deterministic=True, workers=2,
                                     cache='ram', project=OUT, name=name, exist_ok=True)
        open(bitti, 'w').write('ok')
    m = YOLO(f'{run}/weights/best.pt')
    r = m.val(data=yaml_yol, split='val', imgsz=960, batch=8, plots=False, verbose=False)
    return dict(deney=ad, seed=seed, P=r.box.mp, R=r.box.mr,
                mAP50=r.box.map50, mAP50_95=r.box.map)

csv = f'{BASE}/results_tarama_val.csv'
tablo = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame(
    columns=['deney', 'seed', 'P', 'R', 'mAP50', 'mAP50_95'])
for seed in SEEDLER:
    for ad in DENEYLER:
        s = egit(ad, seed)
        print(s)
        tablo = pd.concat([tablo[~((tablo.deney == ad) & (tablo.seed == seed))],
                           pd.DataFrame([s])])
        tablo.to_csv(csv, index=False)
print(tablo)

In [ ]:
import os, shutil, re
import pandas as pd
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
OUT  = f'{BASE}/runs_tarama'
os.makedirs(OUT, exist_ok=True)
DENEYLER = ['bm3d_s30']
SEEDLER  = [0, 1, 2]

def hazirla(ad):
    src, dst = f'{BASE}/dataset_{ad}', f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    txt = re.sub(r'^path:.*$', f'path: {dst}', open(f'{src}/data.yaml').read(), flags=re.M)
    open(f'{dst}/data.yaml', 'w').write(txt)
    return f'{dst}/data.yaml'

def egit(ad, seed):
    name = f'seed{seed}_{ad}'
    yaml_yol = hazirla(ad)
    run = f'{OUT}/{name}'
    bitti = f'{run}/.done'
    if not os.path.exists(bitti):
        last = f'{run}/weights/last.pt'
        if os.path.exists(last):
            YOLO(last).train(resume=True)
        else:
            YOLO('yolo11s.pt').train(data=yaml_yol, epochs=100, patience=30, imgsz=960,
                                     batch=8, seed=seed, deterministic=True, workers=2,
                                     cache='ram', project=OUT, name=name, exist_ok=True)
        open(bitti, 'w').write('ok')
    m = YOLO(f'{run}/weights/best.pt')
    r = m.val(data=yaml_yol, split='val', imgsz=960, batch=8, plots=False, verbose=False)
    return dict(deney=ad, seed=seed, P=r.box.mp, R=r.box.mr,
                mAP50=r.box.map50, mAP50_95=r.box.map)

csv = f'{BASE}/results_tarama_val.csv'
tablo = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame(
    columns=['deney', 'seed', 'P', 'R', 'mAP50', 'mAP50_95'])
for seed in SEEDLER:
    for ad in DENEYLER:
        s = egit(ad, seed)
        print(s)
        tablo = pd.concat([tablo[~((tablo.deney == ad) & (tablo.seed == seed))],
                           pd.DataFrame([s])])
        tablo.to_csv(csv, index=False)
print(tablo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import cv2, numpy as np
print('OpenCV:', cv2.__version__, '| NumPy:', np.__version__)

In [ ]:
import glob, cv2, numpy as np, os
KOK = '/content/drive/MyDrive/sonar_proje/dataset_orig'
yollar = [p for p in glob.glob(KOK + '/**/*', recursive=True)
          if p.lower().endswith(('.jpg', '.jpeg', '.png')) and '/train' in p.replace('\\','/')]
print('bulunan train görüntüsü:', len(yollar))
yollar = sorted(yollar)[::max(1, len(yollar)//20)][:20]

def komsu_korelasyon(res, d):
    # yatay ve dikey, d piksel kaydırmalı korelasyon
    h = np.corrcoef(res[:, :-d].ravel(), res[:, d:].ravel())[0, 1]
    v = np.corrcoef(res[:-d, :].ravel(), res[d:, :].ravel())[0, 1]
    return h, v

def kalinti(gri):
    gri = gri.astype(np.float32)
    return gri - cv2.GaussianBlur(gri, (0, 0), 2)   # yaklaşık "gürültü" kısmı

sonuc = {d: [] for d in (1, 2, 3)}
for p in yollar:
    gri = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
    res = kalinti(gri)
    for d in sonuc:
        sonuc[d].append(komsu_korelasyon(res, d))

# referans: aynı işlemden geçen gerçekten bağımsız (beyaz) gürültü
beyaz = kalinti(np.random.normal(128, 20, gri.shape).astype(np.float32))
print('\nKomşu piksel korelasyonu (yatay, dikey)')
for d in sonuc:
    a = np.mean(sonuc[d], axis=0)
    r = komsu_korelasyon(beyaz, d)
    print(f'{d} piksel kayma | sonar: {a[0]:.3f}, {a[1]:.3f} | beyaz gürültü referansı: {r[0]:.3f}, {r[1]:.3f}')

In [ ]:
import os, json
d = '/content/drive/MyDrive/sonar_proje'
yollar = [p for p in glob.glob(d + '/dataset_orig/**/*', recursive=True)
          if p.lower().endswith(('.jpg', '.jpeg', '.png')) and '/train' in p]
sp = json.load(open(d + '/split.json'))
print(type(sp), list(sp.keys()) if isinstance(sp, dict) else len(sp))
adlar = {os.path.basename(p) for p in yollar}
for k, v in sp.items():
    print(k, len(v))
print('train klasöründe olup split train listesinde olmayan:',
      adlar - {os.path.basename(x) for x in sp.get('train', [])})

In [ ]:
ex = json.load(open(d + '/excluded_images.json'))
print(ex)
print('train klasöründe olan dışlananlar:', [e for e in (ex if isinstance(ex, list) else ex.keys()) if os.path.basename(e) in adlar])

In [ ]:
adsiz = {os.path.splitext(a)[0] for a in adlar}
print('dışlanan train klasöründe mi:', ex[0] in adsiz)
for k in ('train', 'val', 'test'):
    print(k, 'split listesinde mi:', any(ex[0] in x for x in sp[k]))

In [ ]:
import glob, numpy as np
d = '/content/drive/MyDrive/sonar_proje'
for p in glob.glob(d + '/dataset_orig/**/*.cache', recursive=True):
    c = np.load(p, allow_pickle=True).item()
    print(p)
    print('  bulunan, eksik etiket, boş, bozuk, toplam =', c['results'])

In [ ]:
import cv2, glob
p = glob.glob(d + '/dataset_orig/**/9bf66ea1-Screenshot_2025-08-03_15.08.49*', recursive=True)
print(p)
for x in p:
    if x.lower().endswith(('.jpg', '.jpeg', '.png')):
        print('okunabildi mi:', cv2.imread(x) is not None)

In [ ]:
from PIL import Image
import glob
yollar = glob.glob(d + '/dataset_orig/images/train/*')
bozuk = []
for p in yollar:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        bozuk.append((p.split('/')[-1], str(e)[:60]))
print('toplam:', len(yollar), '| bozuk:', len(bozuk))
print(bozuk)

In [ ]:
!pip -q install careamics tifffile
from google.colab import drive
drive.mount('/content/drive')

import cv2, glob, os, random, numpy as np, tifffile
d = '/content/drive/MyDrive/sonar_proje'
HAM = '/content/n2v_egitim_y'          # /content altında, Drive'dan hızlı
os.makedirs(HAM, exist_ok=True)

yollar = sorted(glob.glob(d + '/dataset_orig/images/train/*'))
random.seed(0)
secim = random.sample(yollar, 200)
n_ok, n_bozuk = 0, 0
for p in secim:
    im = cv2.imread(p)
    if im is None:                      # bozuk görüntüyü atla
        n_bozuk += 1
        continue
    y = cv2.cvtColor(im, cv2.COLOR_BGR2YCrCb)[:, :, 0]   # BM3D'deki gibi Y kanalı
    tifffile.imwrite(f'{HAM}/{os.path.splitext(os.path.basename(p))[0]}.tif', y)
    n_ok += 1
print('yazılan:', n_ok, '| bozuk atlanan:', n_bozuk)
print('örnek boyut:', y.shape)

In [ ]:
from careamics.careamist import CAREamist
from careamics.config.factories import create_n2v_config

config = create_n2v_config(
    experiment_name='sonar_n2v', data_type='tiff', axes='YX',
    patch_size=[64, 64], batch_size=64,
    num_epochs=5, num_steps=300,     # epoch başına en çok 300 adım, süreyi sınırlar
)
careamist = CAREamist(config, work_dir='/content/n2v_work')
careamist.train(train_data=HAM)

In [ ]:
import matplotlib.pyplot as plt

vyollar = sorted(glob.glob(d + '/dataset_orig/images/val/*'))[::50][:3]
fig, ax = plt.subplots(3, 3, figsize=(15, 9))
for i, p in enumerate(vyollar):
    y = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2YCrCb)[:, :, 0]
    tahmin = careamist.predict(pred_data=y, axes='YX', data_type='array',
                               tile_size=[256, 256], tile_overlap=[48, 48])
    t = np.squeeze(np.array(tahmin[0] if isinstance(tahmin, (list, tuple)) else tahmin)).astype(np.float32)
    fark = y.astype(np.float32) - t
    # küçük bir kesit (üstten-soldan 400x600) gösterelim
    for j, (im, ad) in enumerate([(y, 'Önce'), (t, 'N2V sonrası'), (fark, 'Fark (önce − sonra)')]):
        ax[i, j].imshow(im[300:700, 600:1200], cmap='gray'); ax[i, j].set_title(ad); ax[i, j].axis('off')
    h = np.corrcoef(fark[:, :-1].ravel(), fark[:, 1:].ravel())[0, 1]
    print(f'{os.path.basename(p)} | fark: ort {fark.mean():.2f}, std {fark.std():.2f} | yatay komşu korelasyonu {h:.3f}')
plt.tight_layout(); plt.show()

In [ ]:
def kalinti(g):
    g = g.astype(np.float32)
    return g - cv2.GaussianBlur(g, (0, 0), 2)

def kor(res, dx, dy):
    if dx:
        return np.corrcoef(res[:, :-dx].ravel(), res[:, dx:].ravel())[0, 1]
    return np.corrcoef(res[:-dy, :].ravel(), res[dy:, :].ravel())[0, 1]

for p in vyollar:
    y = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2YCrCb)[:, :, 0]
    tahmin = careamist.predict(pred_data=y, axes='YX', data_type='array',
                               tile_size=[256, 256], tile_overlap=[48, 48])
    t = np.squeeze(np.array(tahmin[0] if isinstance(tahmin, (list, tuple)) else tahmin)).astype(np.float32)
    ry, rt = kalinti(y), kalinti(t)
    print(os.path.basename(p)[:12],
          '| girdi yatay/dikey:', f'{kor(ry,1,0):.3f}/{kor(ry,0,1):.3f}',
          '| N2V çıktısı yatay/dikey:', f'{kor(rt,1,0):.3f}/{kor(rt,0,1):.3f}',
          '| kalıntı std girdi/çıktı:', f'{ry.std():.2f}/{rt.std():.2f}')

In [ ]:
config2 = create_n2v_config(
    experiment_name='sonar_n2v_uzun', data_type='tiff', axes='YX',
    patch_size=[64, 64], batch_size=64,
    num_epochs=30, num_steps=300,
)
careamist2 = CAREamist(config2, work_dir='/content/n2v_work_uzun')
careamist2.train(train_data=HAM)

!cp -r /content/n2v_work_uzun /content/drive/MyDrive/sonar_proje/n2v_work_uzun

In [ ]:
for p in vyollar:
    y = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2YCrCb)[:, :, 0]
    tahmin = careamist2.predict(pred_data=y, axes='YX', data_type='array',
                                tile_size=[256, 256], tile_overlap=[48, 48])
    t = np.squeeze(np.array(tahmin[0] if isinstance(tahmin, (list, tuple)) else tahmin)).astype(np.float32)
    ry, rt = kalinti(y), kalinti(t)
    print(os.path.basename(p)[:12],
          '| girdi yatay/dikey:', f'{kor(ry,1,0):.3f}/{kor(ry,0,1):.3f}',
          '| N2V çıktısı yatay/dikey:', f'{kor(rt,1,0):.3f}/{kor(rt,0,1):.3f}',
          '| kalıntı std girdi/çıktı:', f'{ry.std():.2f}/{rt.std():.2f}',
          '| fark ort:', f'{(y.astype(np.float32)-t).mean():.2f}')

In [ ]:
import inspect, re
from careamics.config.factories import create_advanced_n2v_config
sig = inspect.signature(create_advanced_n2v_config)
print([p for p in sig.parameters if 'struct' in p], '| default:', {p: sig.parameters[p].default for p in sig.parameters if 'struct' in p})
doc = inspect.getdoc(create_advanced_n2v_config) or ''
for m in re.finditer(r'.*struct.*\n(?:.*\n){0,3}', doc):
    print(m.group(0).rstrip())
try:
    config3 = create_advanced_n2v_config(
        experiment_name='sonar_structn2v', data_type='tiff', axes='YX',
        patch_size=[64, 64], batch_size=64, num_epochs=30, num_steps=300,
        struct_n2v_axes='horizontal', struct_n2v_span=7,
    )
    print('KURULDU')
except Exception as e:
    print('HATA:', type(e).__name__, str(e)[:600])

In [ ]:
careamist3 = CAREamist(config3, work_dir='/content/n2v_work_struct')
careamist3.train(train_data=HAM)
!cp -r /content/n2v_work_struct /content/drive/MyDrive/sonar_proje/n2v_work_struct

In [ ]:
import cv2, numpy as np

def kalinti(g):
    g = g.astype(np.float32)
    return g - cv2.GaussianBlur(g, (0, 0), 2)

def kor(res, dx=0, dy=0):
    h, w = res.shape
    a = res[0:h-dy, 0:w-dx]
    b = res[dy:h, dx:w]
    return np.corrcoef(a.ravel(), b.ravel())[0, 1]

def ac(o):
    while isinstance(o, (list, tuple)):
        o = o[0]
    return np.squeeze(np.asarray(o)).astype(np.float32)

for p in vyollar:
    y = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2YCrCb)[..., 0].astype(np.float32)
    out = careamist3.predict(pred_data=y, axes='YX', data_type='array',
                             tile_size=[256, 256], tile_overlap=[48, 48])
    out = ac(out)
    print('çıktı şekli:', out.shape, '| girdi şekli:', y.shape)
    r_in, r_out = kalinti(y), kalinti(out)
    print(p.split('/')[-1][:20],
          '| yatay kor', f'{kor(r_in,1,0):.3f}->{kor(r_out,1,0):.3f}',
          '| dikey kor', f'{kor(r_in,0,1):.3f}->{kor(r_out,0,1):.3f}',
          '| kalıntı std', f'{r_in.std():.2f}->{r_out.std():.2f}',
          '| ort fark', f'{(out-y).mean():.2f}')

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

def ac(o):
    while isinstance(o, (list, tuple)):
        o = o[0]
    return np.squeeze(np.asarray(o)).astype(np.float32)

def tahmin(cm, y):
    return ac(cm.predict(pred_data=y, axes='YX', data_type='array',
                         tile_size=[256, 256], tile_overlap=[48, 48]))

ROOT = '/content/drive/MyDrive/sonar_proje/dataset_orig'
W, H = 360, 220
fig, ax = plt.subplots(len(vyollar), 3, figsize=(15, 3.6*len(vyollar)))
for i, p in enumerate(vyollar):
    stem = p.split('/')[-1].rsplit('.', 1)[0]
    y = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2YCrCb)[..., 0].astype(np.float32)
    lb = open(f'{ROOT}/labels/val/{stem}.txt').read().split()
    cx, cy = float(lb[1]) * y.shape[1], float(lb[2]) * y.shape[0]
    x0 = int(np.clip(cx - W/2, 0, y.shape[1] - W)); y0 = int(np.clip(cy - H/2, 0, y.shape[0] - H))
    for j, (ad, im) in enumerate([('Girdi', y), ('Düz N2V', tahmin(careamist2, y)),
                                  ('structN2V', tahmin(careamist3, y))]):
        ax[i, j].imshow(im[y0:y0+H, x0:x0+W], cmap='gray', vmin=0, vmax=255)
        ax[i, j].axis('off')
        if i == 0: ax[i, j].set_title(ad)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/sonar_proje/n2v_gorsel_karsilastirma.png', dpi=130)
plt.show()

In [ ]:
import os, json, cv2, tifffile

BASE = '/content/drive/MyDrive/sonar_proje'
HAM_TAM = '/content/n2v_egitim_y_tam'
os.makedirs(HAM_TAM, exist_ok=True)

bad = set(json.load(open(f'{BASE}/excluded_images.json')))
kok = '/content/dataset_orig' if os.path.exists('/content/dataset_orig') else f'{BASE}/dataset_orig'
src = f'{kok}/images/train'

yazilan, atlanan = 0, []
for f in sorted(os.listdir(src)):
    stem = os.path.splitext(f)[0]
    if stem in bad:
        continue
    im = cv2.imread(f'{src}/{f}')
    if im is None:
        atlanan.append(f); continue
    y = cv2.cvtColor(im, cv2.COLOR_BGR2YCrCb)[..., 0]
    tifffile.imwrite(f'{HAM_TAM}/{stem}.tiff', y)
    yazilan += 1
print('yazılan:', yazilan, '| okunamayan:', atlanan)

In [ ]:
from careamics.careamist import CAREamist
from careamics.config.factories import create_n2v_config, create_advanced_n2v_config

HAM_TAM = '/content/n2v_egitim_y_tam'
D = '/content/drive/MyDrive/sonar_proje'

# 1) düz N2V (hocanın istediği referans hat)
cfg_d = create_n2v_config(experiment_name='sonar_n2v_tam', data_type='tiff', axes='YX',
    patch_size=[64, 64], batch_size=64, num_epochs=30, num_steps=300)
cm_d = CAREamist(cfg_d, work_dir='/content/n2v_work_tam')
cm_d.train(train_data=HAM_TAM)
!cp -r /content/n2v_work_tam {D}/n2v_work_tam
print('DÜZ N2V BİTTİ ve Drive a kopyalandı')

# 2) structN2V (yatay maske, span 7)
cfg_s = create_advanced_n2v_config(experiment_name='sonar_structn2v_tam', data_type='tiff', axes='YX',
    patch_size=[64, 64], batch_size=64, num_epochs=30, num_steps=300,
    struct_n2v_axes='horizontal', struct_n2v_span=7)
cm_s = CAREamist(cfg_s, work_dir='/content/n2v_work_struct_tam')
cm_s.train(train_data=HAM_TAM)
!cp -r /content/n2v_work_struct_tam {D}/n2v_work_struct_tam
print('STRUCTN2V BİTTİ ve Drive a kopyalandı')

In [ ]:
import os, json, shutil, cv2, numpy as np
from tqdm import tqdm

BASE = '/content/drive/MyDrive/sonar_proje'
SRC = '/content/dataset_orig' if os.path.exists('/content/dataset_orig') else f'{BASE}/dataset_orig'
bad = set(json.load(open(f'{BASE}/excluded_images.json')))

def ac(o):
    while isinstance(o, (list, tuple)):
        o = o[0]
    return np.squeeze(np.asarray(o)).astype(np.float32)

def uygula(cm, ad, aciklama):
    DST = f'{BASE}/dataset_{ad}'
    json.dump(dict(method=aciklama, kanal='Y (YCrCb), Cr/Cb değişmedi', patch=64, epoch=30, num_steps=300,
                   egitim='722 train görüntüsü'), open(f'{BASE}/{ad}_params.json', 'w'), ensure_ascii=False)
    todo = []
    for s in ['train', 'val', 'test']:
        os.makedirs(f'{DST}/images/{s}', exist_ok=True); os.makedirs(f'{DST}/labels/{s}', exist_ok=True)
        for f in sorted(os.listdir(f'{SRC}/images/{s}')):
            stem = os.path.splitext(f)[0]
            if stem in bad: continue
            todo.append((s, f, stem))
            shutil.copy(f'{SRC}/labels/{s}/{stem}.txt', f'{DST}/labels/{s}/{stem}.txt')
    kalan = [t for t in todo if not os.path.exists(f'{DST}/images/{t[0]}/{t[2]}.png')]
    print(ad, f'| toplam {len(todo)} | kalan {len(kalan)}')
    hatali = []
    for s, f, stem in tqdm(kalan):
        try:
            im = cv2.imread(f'{SRC}/images/{s}/{f}')
            ycc = cv2.cvtColor(im, cv2.COLOR_BGR2YCrCb)
            y = ycc[..., 0].astype(np.float32)
            out = ac(cm.predict(pred_data=y, axes='YX', data_type='array',
                                tile_size=[256, 256], tile_overlap=[48, 48]))
            assert out.shape == y.shape, (out.shape, y.shape)
            ycc[..., 0] = np.clip(out + 0.5, 0, 255).astype(np.uint8)
            cv2.imwrite(f'{DST}/images/{s}/{stem}.png.tmp.png', cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR))
            os.replace(f'{DST}/images/{s}/{stem}.png.tmp.png', f'{DST}/images/{s}/{stem}.png')
        except Exception as e:
            hatali.append((s, f, str(e)[:80])); print('HATA:', s, f, str(e)[:80])
    print(ad, 'bitti | hatalı:', hatali)
    if not hatali:
        if os.path.exists(f'/content/dataset_{ad}'): shutil.rmtree(f'/content/dataset_{ad}')
        shutil.copytree(DST, f'/content/dataset_{ad}')
        open(f'/content/dataset_{ad}/data.yaml', 'w').write(f"path: /content/dataset_{ad}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n")
        print({s: len(os.listdir(f'/content/dataset_{ad}/images/{s}')) for s in ['train', 'val', 'test']})

uygula(cm_d, 'n2v', 'Noise2Void düz (CAREamics v0.2)')
uygula(cm_s, 'n2v_struct', 'structN2V yatay span 7 (CAREamics v0.2)')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

import os, gc, shutil, torch, pandas as pd
from ultralytics import YOLO
assert torch.cuda.is_available(), 'GPU yok -> T4 GPU seç ve tekrar çalıştır'
gpu = torch.cuda.get_device_name(0); print('GPU:', gpu)
if 'T4' not in gpu: print('UYARI: önceki çalışmalar T4 idi, tutarlılık için T4 tercih et')

BASE = '/content/drive/MyDrive/sonar_proje'
DENEYLER = ['n2v']            # ikinci hat için: ['n2v_struct']
SEEDLER = [0, 1, 2]
AUTO_DISCONNECT = True        # iş bitince oturumu kapatır (birim harcamasın)
OUT = f'{BASE}/runs_n2v'; os.makedirs(OUT, exist_ok=True)

def hazirla(ad):
    dst = f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(f'{BASE}/dataset_{ad}', dst)
    open(f'{dst}/data.yaml', 'w').write(
        f"path: {dst}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n")
    print(ad, {s: len(os.listdir(f'{dst}/images/{s}')) for s in ['train', 'val', 'test']})
    return f'{dst}/data.yaml'

def bir_deney(ad, seed, yaml_yol):
    name = f'seed{seed}_{ad}'
    done = f'{OUT}/{name}.done'
    if os.path.exists(done):
        print('atlandı (bitmiş):', name); return
    rdir = f'{OUT}/{name}'
    best, last = f'{rdir}/weights/best.pt', f'{rdir}/weights/last.pt'
    print(f'\n=== {name} başlıyor ===')
    if os.path.exists(last):
        try:
            YOLO(last).train(resume=True)
        except Exception as e:
            print('devam edilemedi (muhtemelen zaten bitmiş):', str(e)[:150])
    else:
        YOLO('yolo11s.pt').train(data=yaml_yol, epochs=100, patience=30, imgsz=960, batch=8,
                                 seed=seed, deterministic=True, workers=2, cache='ram',
                                 project=OUT, name=name, exist_ok=True, plots=False)
    m = YOLO(best); rows = []
    for split in ['val', 'test']:
        r = m.val(data=yaml_yol, split=split, imgsz=960, batch=8, verbose=False, plots=False,
                  project=OUT, name=f'eval_{name}_{split}', exist_ok=True)
        P, R = r.box.mp, r.box.mr
        rows.append(dict(deney=ad, seed=seed, kume=split, precision=P, recall=R,
                         f1=2*P*R/(P+R) if P+R > 0 else 0, mAP50=r.box.map50, mAP50_95=r.box.map))
    csv = f'{BASE}/results_{ad}.csv'
    new = pd.DataFrame(rows).round(4)
    if os.path.exists(csv):
        new = pd.concat([pd.read_csv(csv), new])
    new.drop_duplicates(['deney', 'seed', 'kume'], keep='last').to_csv(csv, index=False)
    open(done, 'w').write('ok')
    del m; gc.collect(); torch.cuda.empty_cache()

for ad in DENEYLER:
    yaml_yol = hazirla(ad)
    for seed in SEEDLER:
        bir_deney(ad, seed, yaml_yol)

cols = ['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']
for ad in DENEYLER:
    d0 = pd.read_csv(f'{BASE}/results_{ad}.csv')
    for kume in ['val', 'test']:
        d = d0[d0.kume == kume]
        print(f'\n== {ad} | {kume} | n={len(d)} ==')
        print(' | '.join(f'{c} {d[c].mean():.3f}±{d[c].std():.3f}' for c in cols))

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

import os, gc, shutil, torch, pandas as pd
from ultralytics import YOLO
assert torch.cuda.is_available(), 'GPU yok -> T4 GPU seç ve tekrar çalıştır'
gpu = torch.cuda.get_device_name(0); print('GPU:', gpu)

BASE = '/content/drive/MyDrive/sonar_proje'
DENEYLER = ['n2v_struct']
SEEDLER = [0, 1, 2]
AUTO_DISCONNECT = True
OUT = f'{BASE}/runs_n2v'; os.makedirs(OUT, exist_ok=True)

def hazirla(ad):
    dst = f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(f'{BASE}/dataset_{ad}', dst)
    open(f'{dst}/data.yaml', 'w').write(
        f"path: {dst}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n")
    print(ad, {s: len(os.listdir(f'{dst}/images/{s}')) for s in ['train', 'val', 'test']})
    return f'{dst}/data.yaml'

def bir_deney(ad, seed, yaml_yol):
    name = f'seed{seed}_{ad}'
    done = f'{OUT}/{name}.done'
    if os.path.exists(done):
        print('atlandı (bitmiş):', name); return
    rdir = f'{OUT}/{name}'
    best, last = f'{rdir}/weights/best.pt', f'{rdir}/weights/last.pt'
    print(f'\n=== {name} başlıyor ===')
    if os.path.exists(last):
        try:
            YOLO(last).train(resume=True)
        except Exception as e:
            print('devam edilemedi (muhtemelen zaten bitmiş):', str(e)[:150])
    else:
        YOLO('yolo11s.pt').train(data=yaml_yol, epochs=100, patience=30, imgsz=960, batch=8,
                                 seed=seed, deterministic=True, workers=2, cache='ram',
                                 project=OUT, name=name, exist_ok=True, plots=False)
    m = YOLO(best); rows = []
    for split in ['val', 'test']:
        r = m.val(data=yaml_yol, split=split, imgsz=960, batch=8, verbose=False, plots=False,
                  project=OUT, name=f'eval_{name}_{split}', exist_ok=True)
        P, R = r.box.mp, r.box.mr
        rows.append(dict(deney=ad, seed=seed, kume=split, precision=P, recall=R,
                         f1=2*P*R/(P+R) if P+R > 0 else 0, mAP50=r.box.map50, mAP50_95=r.box.map))
    csv = f'{BASE}/results_{ad}.csv'
    new = pd.DataFrame(rows).round(4)
    if os.path.exists(csv):
        new = pd.concat([pd.read_csv(csv), new])
    new.drop_duplicates(['deney', 'seed', 'kume'], keep='last').to_csv(csv, index=False)
    open(done, 'w').write('ok')
    del m; gc.collect(); torch.cuda.empty_cache()

for ad in DENEYLER:
    yaml_yol = hazirla(ad)
    for seed in SEEDLER:
        bir_deney(ad, seed, yaml_yol)

cols = ['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']
for ad in DENEYLER:
    d0 = pd.read_csv(f'{BASE}/results_{ad}.csv')
    for kume in ['val', 'test']:
        d = d0[d0.kume == kume]
        print(f'\n== {ad} | {kume} | n={len(d)} ==')
        print(' | '.join(f'{c} {d[c].mean():.3f}±{d[c].std():.3f}' for c in cols))

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

import os, gc, glob, json, cv2, torch, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
def latest(p): return sorted(glob.glob(p), key=os.path.getmtime)[-1]
MODELS = {
    'Orijinal':  (f"{latest(f'{BASE}/runs/baseline_orig*')}/weights/best.pt", 'dataset_orig'),
    'NLM':       (f"{latest(f'{BASE}/runs/train_nlm*')}/weights/best.pt",     'dataset_nlm'),
    'BM3D':      (f"{latest(f'{BASE}/runs/train_bm3d*')}/weights/best.pt",    'dataset_bm3d'),
    'Düz N2V':   (f'{BASE}/runs_n2v/seed0_n2v/weights/best.pt',               'dataset_n2v'),
    'structN2V': (f'{BASE}/runs_n2v/seed0_n2v_struct/weights/best.pt',        'dataset_n2v_struct'),
}
for n, (wp, _) in MODELS.items():
    assert os.path.exists(wp), f'{n} için ağırlık yok: {wp}  (YOLO eğitimi bitti mi?)'
test_files = json.load(open(f'{BASE}/split.json'))['test']

def img_path(name, f):
    root = MODELS[name][1]
    return f'{BASE}/{root}/images/test/' + (f if root == 'dataset_orig' else os.path.splitext(f)[0] + '.png')

gts = []   # (dosya, normalize xyxy, alan)
for f in test_files:
    for line in open(f'{BASE}/dataset_orig/labels/test/{os.path.splitext(f)[0]}.txt').read().splitlines():
        c, cx, cy, w, h = map(float, line.split()[:5])
        gts.append((f, np.array([cx-w/2, cy-h/2, cx+w/2, cy+h/2]), w*h))
small = list(np.argsort([g[2] for g in gts])[:max(1, len(gts)//4)])    # en küçük %25
print('toplam hedef:', len(gts), '| küçük hedef sayısı:', len(small))

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0

preds = {}
for name, (wp, root) in MODELS.items():
    print('tahmin yapılıyor:', name)
    m = YOLO(wp); preds[name] = {}
    for f in test_files:                       # tek tek, batch=1 (bellek taşmasın)
        with torch.no_grad():
            r = m.predict(img_path(name, f), imgsz=960, conf=0.25, batch=1, verbose=False)[0]
        preds[name][f] = [(b.xyxyn[0].cpu().numpy(), float(b.conf[0])) for b in r.boxes]
    del m; gc.collect(); torch.cuda.empty_cache()

def hit(name, g):
    return any(iou(g[1], p) >= 0.5 for p, _ in preds[name][g[0]])

print(f"\n{'Model':10} {'Küçük hedef':>14} {'Tüm hedef':>12} {'Yanlış alarm':>13}")
for name in MODELS:
    hs = [hit(name, g) for g in gts]
    fp = sum(1 for f in test_files for p, c in preds[name][f]
             if not any(iou(g[1], p) >= 0.5 for g in gts if g[0] == f))
    print(f"{name:10} {sum(hs[i] for i in small):>8}/{len(small):<5} {sum(hs):>6}/{len(gts):<5} {fp:>13}")

flags = {i: [hit(n, gts[i]) for n in MODELS] for i in small}
disagree = [i for i in small if len(set(flags[i])) > 1]          # modellerin ayrıştığı hedefler öne
chosen = (disagree + [i for i in small if i not in disagree])[:6]

def to_px(b, W, H, x0, y0):
    return [int(v) for v in (b * np.array([W, H, W, H]) - np.array([x0, y0, x0, y0]))]

CW, CH = 480, 300
fig, ax = plt.subplots(len(chosen), len(MODELS), figsize=(4.2*len(MODELS), 2.8*len(chosen)))
ax = np.atleast_2d(ax)
for ri, gi in enumerate(chosen):
    f, box, _ = gts[gi]
    for ci, name in enumerate(MODELS):
        im = cv2.imread(img_path(name, f)); H, W = im.shape[:2]
        cx, cy = (box[0]+box[2])/2*W, (box[1]+box[3])/2*H
        x0 = int(np.clip(cx - CW/2, 0, max(W-CW, 0))); y0 = int(np.clip(cy - CH/2, 0, max(H-CH, 0)))
        crop = im[y0:y0+CH, x0:x0+CW].copy()
        g = to_px(box, W, H, x0, y0)
        cv2.rectangle(crop, (g[0], g[1]), (g[2], g[3]), (0, 255, 0), 2)          # yeşil = gerçek
        for p, conf in preds[name][f]:
            q = to_px(p, W, H, x0, y0)
            cv2.rectangle(crop, (q[0], q[1]), (q[2], q[3]), (0, 0, 255), 2)       # kırmızı = tahmin
            cv2.putText(crop, f'{conf:.2f}', (q[0], max(q[1]-4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        ax[ri, ci].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax[ri, ci].axis('off')
        if ri == 0: ax[ri, ci].set_title(name)
fig.suptitle('Yeşil: gerçek kutu | Kırmızı: tahmin (güven ≥ 0.25) | Seed 0 modelleri, test kümesi (sadece rapor)')
plt.tight_layout(); plt.savefig(f'{BASE}/kucuk_hedef_karsilastirma_n2v.png', dpi=130); plt.show()

from google.colab import runtime
runtime.unassign()   # bitince oturumu kapat